In [18]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [19]:
import inspect
print(inspect.getsource(train))

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids,
                        attention_mask = source_mask,
                        decoder_input_ids = decoder_input_ids,
                        labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    return train_loss



In [20]:
import numpy as np
from datasets import load_dataset

news = load_dataset("argilla/news-summary", split = "test")
df = news.to_pandas().sample(5000, random_state = 42)[["text", "prediction"]]
df["text"] = "summarize: " + df["text"]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])
train, valid, test = np.split(df.sample(frac = 1, random_state = 42),
                              [int(0.6 * len(df)), int(0.8 * len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Putin says had useful interaction with Trump at Vi
3000
1000
1000


c:\Users\KDT-17\Documents\17-NLP & CV\.venv\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [21]:
import torch

from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    source = tokenizer(text = data.text.tolist(),
                       padding = "max_length",
                       max_length = 128,
                       pad_to_max_length = True,
                       truncation = True,
                       return_tensors = "pt")
    
    target = tokenizer(text = data.prediction.tolist(),
                       padding = "max_length",
                       max_length = 128,
                       pad_to_max_length = True,
                       truncation = True,
                       return_tensors = "pt")

    source_ids = source['input_ids'].squeeze().to(device)
    source_mask = source['attention_mask'].squeeze().to(device)

    target_ids = target['input_ids'].squeeze().to(device)
    target_mask = target['attention_mask'].squeeze().to(device)

    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler = data_sampler, batch_size = batch_size)

    return dataloader

In [22]:
epochs = 5
batch_size = 8

tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path = "t5-small")

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

[tensor([[21603,    10,   549,  ...,     6,   713,     1],
        [21603,    10,   549,  ...,  7821,  8616,     1],
        [21603,    10, 10264,  ...,    12,   161,     1],
        ...,
        [21603,    10,     3,  ...,  2753,     5,     1],
        [21603,    10,  7933,  ...,   925,    17,     1],
        [21603,    10,  7109,  ...,     0,     0,     0]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), tensor([[ 2526,    63,  4299,  ...,     0,     0,     0],
        [ 9487, 13591,    13,  ...,     0,     0,     0],
        [ 1626,    89,  2046,  ...,     0,     0,     0],
        ...,
        [26499,     6, 16523,  ...,     0,     0,     0],
        [24463,    10,  2678,  ...,     0,     0,     0],
        [ 4263, 20417,  5873,  ...,     0,     0,     0]], device='cuda:0'), ten

In [23]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path = "t5-small").to(device)
optimizer = optim.AdamW(model.parameters(), lr = 1e-5, eps = 1e-8)

In [24]:
def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis = 1).flatten()
    labels_flat = labels.flatten()

    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask = source_mask,
                        decoder_input_ids = decoder_input_ids, labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)

    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(input_ids = source_ids, attention_mask = source_mask,
                            decoder_input_ids = decoder_input_ids, labels = labels)
            loss = outputs.loss
            val_loss += loss.item()

        val_loss = val_loss / len(dataloader)

    return val_loss

In [25]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)

    print(epoch + 1, train_loss, val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "./models/T5ForConditionalGeneration.pt")
        print("Saved!!")

1 4.3117808183034265 3.3316776332855222
Saved!!
2 3.421152669906616 2.914762886047363
Saved!!
3 3.134090841293335 2.768085256576538
Saved!!
4 2.997528755187988 2.678118063926697
Saved!!
5 2.8907149016062417 2.613957956314087
Saved!!


In [26]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        generated_ids = model.generate(input_ids = source_ids, attention_mask = source_mask,
                                       max_length = 128, num_beams = 3, repetition_penalty = 2.5,
                                       length_penalty = 1.0, early_stopping = True)
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(generated, skip_special_tokens = True,
                                    clean_up_tokenization_spaces = True)
            actual = tokenizer.decode(target, skip_special_tokens = True,
                                      clean_up_tokenization_tokens = True)
            print(pred)
            print(actual)

        break

Clinton leads Trump by 4 percentage points in four-war race for Nov. 8 election
Clinton leads Trump by 4 points in Washington Post: ABC News poll
U.S. senators sharpen line of attack against Gorsuch's nomination to Supreme Court
Democrats question independence of Trump Supreme Court nominee
U.S. warns Saudi Arabia over Yemen's humanitarian situation could constrain U.S. aid, a U.S. official says.
In push for Yemen aid, U.S. warned Saudis of threats in Congress
Romanian anti-corruption prosecutors open investigation into Liviu Dragnea on suspicion of forming criminal group to siphon off cash from state projects
Romanian ruling party leader investigated over 'criminal group'
environmental activist endorsed Hillary Clinton for U.S. president
Billionaire environmental activist Tom Steyer endorses Clinton
tv presenter delivers news of Pyongyang nuclear test with her usual gusto.
Voice of triumph or doom: North Korean presenter back in limelight for nuclear test
Delson Guarate, Yon Goicoeche